# Pareja A · Idea 2 — Leer el texto del cartel (OCR)

EasyOCR lee las letras del cartel; el codificador de texto de CLIP (`codificar_texto`, máx. 77 tokens) las vuelve vector, que se concatena al de la imagen. Si no hay texto: vector de ceros (u otra decisión, documentada). **Primero con 2 mil carteles**; si ayuda, todo el dataset.

**Dónde se programa:** `codificadores.py`: la variante `ocr` (y agregar `easyocr` a `requirements.txt`). La lógica va en los `.py` (se revisa en el Pull Request); este
notebook solo la corre. **Listo cuando** su prueba pasa (quiten el `xfail` de `test_idea2` en
`tests/test_pendientes.py`) y el resultado en validación está en el registro.

Antes, una vez para todo el equipo: `00_reproducir_base` (la base `v0.1.0`).

## Preparar el entorno

**GPU (T4)**: *Entorno de ejecución → Cambiar tipo de entorno de ejecución*. La cuota gratis se acaba: calculen primero lo prioritario y repartan el cómputo entre sus cuentas.

- `P2M_DATOS`: dataset compartido (solo lectura). `P2M_SALIDAS`: carpeta **compartida del equipo**
  (`poster2movie_clasificador`), donde quedan vectores, modelos y predicciones para las otras parejas.
  Agreguen las dos a *Mi unidad* como acceso directo.
- Trabaja en la rama **`A-ocr`** (se crea desde `main` si no existe).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.environ["P2M_DATOS"] = "/content/drive/MyDrive/poster2movie_datos"            # solo lectura
os.environ["P2M_SALIDAS"] = "/content/drive/MyDrive/poster2movie_clasificador"   # compartida del equipo
REPO = "11909/Poster2Movie-Clasificador"
RAMA = "A-ocr"
DESTINO = "/content/clasificador"
if not os.path.isdir(DESTINO):
    !git clone -q --recurse-submodules https://github.com/{REPO}.git {DESTINO}
%cd {DESTINO}
!git fetch -q origin
!git checkout -q {RAMA} 2>/dev/null || git checkout -q -b {RAMA} origin/main
!git pull -q origin {RAMA} 2>/dev/null; git submodule update -q --init
!pip install -q -r requirements.txt
!nvidia-smi -L

In [ ]:
!pip install -q easyocr

In [ ]:
# ¿ya pasa la prueba de la idea? (mientras esté pendiente sale como "xfailed")
!python -m pytest -q tests/test_pendientes.py -k idea2 -rxX

## 1. Prueba chica: 2 mil carteles fijos

`--muestra 2000` toma siempre las mismas 2 mil películas (de todos los splits). **Ojo:** el
experimento solo ve esas (~1,600 de train, ~200 de val): el número es ruidoso. Sirve para decidir
si vale la pena correr todo el dataset, no para publicar.

In [ ]:
!python -m clasificador.codificadores clip_b32 --variante ocr --muestra 2000

In [ ]:
# entrenar en train, evaluar en VALIDACIÓN y comparar contra base (los 4 débiles salen primero)
for nombre in ["A2_ocr"]:
    !python -m clasificador.entrenar experimentos/{nombre}.yaml
    !python -m clasificador.evaluar {nombre}
    !python -m clasificador.comparar base {nombre}

## 2. Si ayuda: todo el dataset (varias horas de GPU; repartan entre cuentas)

In [ ]:
# !python -m clasificador.codificadores clip_b32 --variante ocr
# luego vuelvan a correr la celda de entrenar/evaluar

## Subir a la rama `A-ocr`

En Colab el repo se borra al cerrar: suban `resultados/` antes. Se sube a **`A-ocr`**, nunca a
`main`; luego abran el Pull Request (lo revisa alguien de **otra** pareja). Necesitan un token de
GitHub con permiso de escritura (*Settings → Developer settings → Personal access tokens → Tokens (classic)*, con el permiso `repo` (los *fine-grained* no sirven: el repo es de otra cuenta personal)).

In [ ]:
from getpass import getpass
!git config user.name "{input('Tu nombre: ')}"
!git config user.email "{input('Tu correo de GitHub: ')}"
!git add experimentos resultados clasificador tests
!git commit -q -m "{RAMA}: {input('Qué hiciste (mensaje del commit): ')}"
token = getpass("Token de GitHub: ")
!git push -q https://{token}@github.com/{REPO}.git HEAD:{RAMA}
del token
print(f"Abre el Pull Request: https://github.com/{REPO}/compare/main...{RAMA}")